# Data Collection from yfinance

## Load needed Libraries and modules

In [1]:
# import Libs
import pandas as pd
import datetime as dt
import pytz
import os
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# import modules
from datetime import datetime, timezone
from datetime import date, time
from math import trunc
from dateutil.parser import parse

## Collecting Options Data from Yahoo Finance using "yfinance"

### Getting AAPL underlying stock prices

In [2]:
# Define the ticker symbol
tickerSymbol = 'AAPL'

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)

# Get the historical prices for this ticker
mb_tickerdf12302025 = tickerData.history(period='5y')

#Convert index to date only
mb_tickerdf12302025.index = mb_tickerdf12302025.index.date

# See the data
mb_tickerdf12302025

,Open,High,Low,Close,Volume,Dividends,Stock Splits
2020-12-31,130.520487,131.162969,128.223139,129.167389,99116600,0.0,0.0
2021-01-04,129.975370,130.062977,123.394829,125.974480,143301900,0.0,0.0
2021-01-05,125.468261,128.242606,125.020466,127.531975,97664900,0.0,0.0
2021-01-06,124.329352,127.570950,123.024922,123.239082,155088000,0.0,0.0
2021-01-07,124.952354,128.135547,124.465627,127.444389,109578200,0.0,0.0
...,...,...,...,...,...,...,...
2025-12-23,270.839996,272.500000,269.559998,272.359985,29642000,0.0,0.0
2025-12-24,272.339996,275.429993,272.200012,273.809998,17910600,0.0,0.0
2025-12-26,274.160004,275.369995,272.859985,273.399994,21521800,0.0,0.0
2025-12-29,272.690002,274.359985,272.350006,273.760010,23715200,0.0,0.0


In [3]:
mb_tickerdf12302025.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1255 entries, 2020-12-31 to 2025-12-30
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1255 non-null   float64
 1   High          1255 non-null   float64
 2   Low           1255 non-null   float64
 3   Close         1255 non-null   float64
 4   Volume        1255 non-null   int64  
 5   Dividends     1255 non-null   float64
 6   Stock Splits  1255 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 78.4+ KB


### Saving the Stock Prices dataframe into a csv file

In [4]:
# get current date and time
current_datetime = datetime.now()
print("Current date & time : ", current_datetime)
filename1 = datetime.now().strftime("%Y-%m-%d %H-%M-%S-%f")

# create a file object along with extension
file_name = "mb_tickerdf12302025 "+filename1+".csv"
#print (file_name)

# converting to csv
mb_tickerdf12302025.to_csv(file_name)
print("~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~")
print("File saved named:")
print (file_name)
print(current_datetime)    
print("~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~")

Current date & time :  2025-12-30 17:24:53.423058
~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~
File saved named:
mb_tickerdf12302025 2025-12-30 17-24-53-423976.csv
2025-12-30 17:24:53.423058
~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~


### Getting AAPL stock Options related data including Implied Volatilities

In [5]:
# Define the ticker symbol
tickerSymbol = 'AAPL'

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)

# Get the option expirations
options_expirations = tickerData.options

# Prepare a list to hold all options data
options_list = []

# Loop through each options expiration date
for expiration in options_expirations:
    # Get options data for each expiration date
    options_data = tickerData.option_chain(expiration)
    
    # Add option type and expiration date to the data
    options_data.calls['OptionType'] = 'Call'
    options_data.calls['expirationDate'] = expiration
    options_data.puts['OptionType'] = 'Put'
    options_data.puts['expirationDate'] = expiration
    
    # Append the data to the options_list
    options_list.append(options_data.calls)
    options_list.append(options_data.puts)

# Concatenate all options data into a single DataFrame
mb_all_options_data12302025 = pd.concat(options_list)

# Reset the index and drop the old one
mb_all_options_data12302025.reset_index(drop=True, inplace=True)

# Convert the expirationDate column to datetime
mb_all_options_data12302025['expirationDate'] = pd.to_datetime(mb_all_options_data12302025['expirationDate'])

# Calculate the time to expiration in days
valuation_date = pd.Timestamp("2025-12-30")
mb_all_options_data12302025['time_to_expiration'] = (
    mb_all_options_data12302025['expirationDate'] - valuation_date
).dt.days


# Display the DataFrame
mb_all_options_data12302025

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,OptionType,expirationDate,time_to_expiration
0,AAPL260102C00120000,2025-12-29 14:52:51+00:00,120.0,153.88,151.60,154.95,0.000000,0.000000,1.0,1,3.335939,True,REGULAR,USD,Call,2026-01-02,3
1,AAPL260102C00135000,2025-12-08 14:31:30+00:00,135.0,144.95,136.55,139.95,0.000000,0.000000,NaN,2,2.843753,True,REGULAR,USD,Call,2026-01-02,3
2,AAPL260102C00145000,2025-11-19 16:06:45+00:00,145.0,126.05,126.55,130.00,0.000000,0.000000,NaN,6,2.621097,True,REGULAR,USD,Call,2026-01-02,3
3,AAPL260102C00150000,2025-12-29 20:59:30+00:00,150.0,123.83,121.60,124.95,0.000000,0.000000,2.0,12,2.492191,True,REGULAR,USD,Call,2026-01-02,3
4,AAPL260102C00155000,2025-12-29 20:59:30+00:00,155.0,118.90,116.55,120.00,0.000000,0.000000,4.0,3,2.369145,True,REGULAR,USD,Call,2026-01-02,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2317,AAPL280317P00330000,2025-12-26 19:28:14+00:00,330.0,65.85,66.55,67.40,0.000000,0.000000,2.0,9,0.174355,True,REGULAR,USD,Put,2028-03-17,808
2318,AAPL280317P00340000,2025-12-30 17:32:03+00:00,340.0,74.42,73.70,74.50,1.199997,1.638892,2.0,1,0.165444,True,REGULAR,USD,Put,2028-03-17,808
2319,AAPL280317P00350000,2025-12-30 17:34:24+00:00,350.0,81.91,80.45,82.70,-0.369995,-0.449678,2.0,201,0.161980,True,REGULAR,USD,Put,2028-03-17,808
2320,AAPL280317P00380000,2025-12-22 14:30:05+00:00,380.0,107.46,106.30,108.45,0.000000,0.000000,200.0,201,0.140023,True,REGULAR,USD,Put,2028-03-17,808


In [6]:
mb_all_options_data12302025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2322 entries, 0 to 2321
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   contractSymbol      2322 non-null   object             
 1   lastTradeDate       2322 non-null   datetime64[ns, UTC]
 2   strike              2322 non-null   float64            
 3   lastPrice           2322 non-null   float64            
 4   bid                 2322 non-null   float64            
 5   ask                 2322 non-null   float64            
 6   change              2322 non-null   float64            
 7   percentChange       2322 non-null   float64            
 8   volume              2239 non-null   float64            
 9   openInterest        2322 non-null   int64              
 10  impliedVolatility   2322 non-null   float64            
 11  inTheMoney          2322 non-null   bool               
 12  contractSize        2322 non-null 

In [7]:
mb_all_options_data12302025.describe().transpose()

,count,mean,min,25%,50%,75%,max,std
strike,2322.0,234.918174,5.0,155.0,235.0,310.0,550.0,113.381492
lastPrice,2322.0,46.212976,0.01,0.55,12.05,72.9425,275.8,64.061545
bid,2322.0,45.206154,0.0,0.45,11.425,70.9125,275.05,63.353561
ask,2322.0,46.316331,0.0,0.5625,11.625,72.625,278.9,64.645591
change,2322.0,-0.12084,-13.75,-0.06,0.0,0.0,6.029999,0.733023
percentChange,2322.0,-0.639302,-88.181816,-1.276371,0.0,0.0,700.0001,25.146162
volume,2239.0,152.047343,1.0,2.0,5.0,26.0,38332.0,1239.952175
openInterest,2322.0,2175.37683,0.0,42.0,268.5,1614.75,86977.0,6115.190455
impliedVolatility,2322.0,0.500941,0.00001,0.249161,0.330756,0.548222,9.960941,0.551448
expirationDate,2322,2026-09-25 01:58:26.976744192,2026-01-02 00:00:00,2026-02-20 00:00:00,2026-06-18 00:00:00,2027-01-15 00:00:00,2028-03-17 00:00:00,NaN


### Saving Options Information dataframe into a csv file

In [8]:
# get current date and time
current_datetime = datetime.now()
print("Current date & time : ", current_datetime)
filename1 = datetime.now().strftime("%Y-%m-%d %H-%M-%S-%f")

# create a file object along with extension
file_name = "mb_all_options_data12302025 "+filename1+".csv"
#print (file_name)

# converting to csv
mb_all_options_data12302025.to_csv(file_name)
print("~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~")
print("File saved named:")
print (file_name)
print(current_datetime)    
print("~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~")

Current date & time :  2025-12-30 17:24:55.033511
~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~
File saved named:
mb_all_options_data12302025 2025-12-30 17-24-55-033746.csv
2025-12-30 17:24:55.033511
~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~
